# Chuẩn bị môi trường

In [1]:
import os
import csv
import cv2
import mediapipe as mp
from tqdm import tqdm
import matplotlib.pyplot as plt


# Tải dataset

In [ ]:
DATASET_DIR = "../Dts/data"          
OUTPUT_CSV = "../Output/face_landmarksv.csv"
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

mp_face_mesh = mp.solutions.face_mesh

NUM_LANDMARKS = 478

# Đánh số 478 toạ độ

In [3]:
def danhsotoado():
    header = []
    header.append("filepath")
    for i in range(NUM_LANDMARKS):
        header += [f"x{i}", f"y{i}", f"z{i}"]
    header.append("label")
    return header

# Trích xuất 478 toạ độ từ ảnh

In [4]:
def trich_xuat_toa_do_tu_anh(face_mesh, image_path):
    """Trả về list 1434 giá trị (x,y,z * 478) hoặc None nếu không detect được mặt."""
    image = cv2.imread(image_path)
    if image is None:
        return None

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 
    # CV2 đọc ảnh theo Blue-Green-Red, nhưng mà Mediapipe đọc theo Red-Green-Blue dẫn đến quá trình trích xuất sai lệch.
    # Do vậy cần phải convert ảnh từ BGR sang RGB để trích xuất đúng.
    results = face_mesh.process(image_rgb)

    if not results.multi_face_landmarks: # Không tìm được mặt thì return None
        return None

    face_landmarks = results.multi_face_landmarks[0] # Nếu detect từ 2 mặt trở lên thì lấy cái mặt được detect đầu tiên để gắn toạ độ

    row = []
    for lm in face_landmarks.landmark:
        row.extend([lm.x, lm.y, lm.z]) # Lấy toạ độ (x,y,z) của 478 điểm

    return row

# Gắn nhãn cho mỗi ảnh

In [5]:
def load_label_id(class_id_path):
    """
    Đọc file Class_ID.txt dạng:
        0: angry
        1: disgust
        2: fear
        ...
    Trả về dict: {"angry": 0, "disgust": 1, "fear": 2, ...}
    """
    name_to_id = {}
    with open(class_id_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()          # bỏ khoảng trắng/xuống dòng thừa
            if not line:
                continue                 # bỏ qua dòng trống
            id_str, name = line.split(":", 1)   # tách theo dấu ":"
            class_id = int(id_str.strip())      # "0" → 0 (số nguyên)
            class_name = name.strip()           # " angry" → "angry"
            name_to_id[class_name] = class_id

    return name_to_id

In [6]:
header = danhsotoado()
class_mapping = load_label_id("Class_ID.txt")
rows_written = 0
rows_skipped = 0
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
with mp_face_mesh.FaceMesh(
    static_image_mode=True,      # bắt buộc True vì xử lý ảnh tĩnh, không phải xử lí theo thời gian thực
    max_num_faces=1,
    refine_landmarks=True,       # bật thêm landmark quanh mắt/môi cho chính xác hơn
    min_detection_confidence=0.5,
) as face_mesh, open(OUTPUT_CSV, "w", newline="") as f_out:

    writer = csv.writer(f_out)
    writer.writerow(header) # Tạo hàng tiêu đề cho file CSV

    labels = sorted(
        d for d in os.listdir(DATASET_DIR)
        if os.path.isdir(os.path.join(DATASET_DIR, d))
        if d.lower() in class_mapping
    )
    print(f"Tìm thấy {len(labels)} nhãn: {labels}")
    scan_fail =[]
    scan_success =[]
    for label in labels:
        label_dir = os.path.join(DATASET_DIR, label)
        image_files = [
            fn for fn in os.listdir(label_dir)
            if fn.lower().endswith(IMAGE_EXTENSIONS)
        ]
        fail =0
        sucess =0
        for fn in tqdm(image_files, desc=f"Đang xử lý '{label}'"):
            image_path = os.path.join(label_dir, fn)
            landmarks = trich_xuat_toa_do_tu_anh(face_mesh, image_path)

            if landmarks is None:
                rows_skipped += 1
                fail+=1
                continue
            label_id = class_mapping.get(label, -1)  # mặc định -1 nếu không có trong mapping
            relative_path = os.path.join(label, fn)
            writer.writerow([relative_path] + landmarks + [label_id])
            rows_written += 1
            sucess +=1
        scan_fail.append(fail)
        scan_success.append(sucess)
print(f"\nXong! Ghi được {rows_written} dòng vào '{OUTPUT_CSV}'.")
print(f"Bỏ qua {rows_skipped} ảnh (không detect được mặt).")

FileNotFoundError: [WinError 3] The system cannot find the path specified: '../Dts/emote'

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)

print(df.head(1).shape)
display(df.head())

(1, 1436)


,filepath,x0,y0,z0,x1,y1,z1,x2,y2,z2,...,x475,y475,z475,x476,y476,z476,x477,y477,z477,label
0,angry\angry1.jpg,0.459758,0.777743,-0.059348,0.491621,0.671920,-0.218034,0.490570,0.692551,-0.086356,...,0.759845,0.373553,-0.044700,0.704184,0.408325,-0.044700,0.753082,0.450641,-0.044700,0
1,angry\angry10.jpg,0.364415,0.584031,-0.082794,0.222786,0.500227,-0.179605,0.276865,0.546710,-0.096145,...,0.638469,0.285548,-0.089178,0.590634,0.310230,-0.089178,0.623666,0.352124,-0.089178,0
2,angry\angry100.jpg,0.547032,0.789640,-0.097028,0.585846,0.655865,-0.235331,0.563893,0.680321,-0.114041,...,0.894488,0.414611,0.033070,0.855359,0.442410,0.033070,0.893590,0.478173,0.033070,0
3,angry\angry1000.jpg,0.705499,0.699734,-0.084416,0.759170,0.578872,-0.189502,0.720096,0.603527,-0.089691,...,0.870490,0.330406,0.088703,0.851800,0.352465,0.088703,0.880078,0.374910,0.088703,0
4,angry\angry1001.jpg,0.405940,0.759520,-0.095442,0.407376,0.636814,-0.207587,0.411611,0.662889,-0.098002,...,0.663642,0.361715,-0.013399,0.614248,0.388299,-0.013399,0.649929,0.421193,-0.013399,0


# Xuất báo cáo thống kê, biểu đồ phân bố nhãn

In [ ]:
for idx, label in enumerate(labels):
    print(f"Số lượng bỏ qua của {label} là {scan_fail[idx]}")
for idx, label in enumerate(labels):
    print(f"Số lượng quét được của {label} là {scan_success[idx]}")


Số lượng bỏ qua của angry là 64
Số lượng bỏ qua của disgust là 30
Số lượng bỏ qua của fear là 184
Số lượng bỏ qua của happy là 39
Số lượng bỏ qua của neutral là 47
Số lượng bỏ qua của sad là 118
Số lượng bỏ qua của surprise là 28
Số lượng quét được của angry là 1710
Số lượng quét được của disgust là 1348
Số lượng quét được của fear là 2716
Số lượng quét được của happy là 1883
Số lượng quét được của neutral là 1788
Số lượng quét được của sad là 2614
Số lượng quét được của surprise là 1826


In [ ]:
plt.figure(figsize=(10,6))

plt.bar(labels, scan_fail, label='Fail',color="navy", alpha=1)
plt.bar(labels, scan_success, label='Success',color="blue", alpha=0.2)

plt.xlabel('Emotion')
plt.ylabel('Count')
plt.title('Bar Chart')
plt.legend()
bar1 = plt.bar(labels, scan_fail, alpha=0)
bar2 = plt.bar(labels, scan_success, alpha=0)
plt.tight_layout()
plt.bar_label(bar1, padding=3)
plt.bar_label(bar2, padding=3)
plt.savefig('bar_chart.png', dpi=300)
plt.close()